In [3]:
# ============================================================
# 05.05 — MULTIMODAL FUSION
# ============================================================
#
# Purpose:
# Reproduce and validate the existing multimodal
# knowledge-level weighted probability fusion baseline.
#
# Pipeline:
#
# Genetic 32-D       -> Genetic expert       -> P_G
# Behavioral 32-D   -> Behavioral expert    -> P_B
# Environmental 32-D -> Environmental expert -> P_E
#
#                         |
#                         v
#
#             Weighted Evidence Fusion
#
#                         |
#                         v
#
#                   Final Probability
#
#                         |
#                         v
#
#                  Threshold = 0.50
#
#                         |
#                         v
#
#                    ASD / NON-ASD
#
# Important:
# - No model training
# - No checkpoint modification
# - No patient-level row matching
# - No row-wise concatenation
# - No cross-modal relationship probability used here
# ============================================================

print("=" * 80)
print("05.05 — MULTIMODAL FUSION")
print("=" * 80)

print("\nObjective:")
print("Reproduce and validate the project's weighted probability-level")
print("multimodal evidence fusion baseline.")

print("\nTraining:")
print("NO TRAINING")

print("\nCheckpoint modification:")
print("NONE")

print("\nFusion type:")
print("Knowledge-level weighted probability evidence fusion")

print("\nDecision threshold:")
print("0.50")

print("\nStatus:")
print("READY")

05.05 — MULTIMODAL FUSION

Objective:
Reproduce and validate the project's weighted probability-level
multimodal evidence fusion baseline.

Training:
NO TRAINING

Checkpoint modification:
NONE

Fusion type:
Knowledge-level weighted probability evidence fusion

Decision threshold:
0.50

Status:
READY


In [4]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path

print("NumPy version :", np.__version__)
print("Pandas version:", pd.__version__)

print("\nImports completed successfully.")

NumPy version : 2.2.6
Pandas version: 2.2.3

Imports completed successfully.


In [5]:
# ============================================================
# CELL 3 — PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

PHASE05_ROOT = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "05_multimodal_integration"
)

REPORT_ROOT = (
    PHASE05_ROOT
    / "reports"
    / "05_05_multimodal_fusion"
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# Existing multimodal fusion artifacts
EXISTING_FUSION_ROOT = (
    PROJECT_ROOT
    / "05_multimodal_fusion"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nPhase 05 root:")
print(PHASE05_ROOT)

print("\nOfficial report directory:")
print(REPORT_ROOT)

print("\nExisting fusion artifact directory:")
print(EXISTING_FUSION_ROOT)

Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Phase 05 root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration

Official report directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_05_multimodal_fusion

Existing fusion artifact directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion


In [6]:
# ============================================================
# CELL 4 — EXISTING ARTIFACT PATHS
# ============================================================

WEIGHTS_FILE = (
    EXISTING_FUSION_ROOT
    / "M3_1_fusion_weights.csv"
)

THREE_MODAL_RESULTS_FILE = (
    EXISTING_FUSION_ROOT
    / "M4_final_P001_P005_three_modal_fusion_results.csv"
)

FLEXIBLE_RESULTS_FILE = (
    EXISTING_FUSION_ROOT
    / "M4_final_P001_P005_flexible_multimodal_results.csv"
)

FINAL_CONFIG_FILE = (
    EXISTING_FUSION_ROOT
    / "M3_2_final_inference_config.json"
)

THREE_MODAL_REPORT_FILE = (
    EXISTING_FUSION_ROOT
    / "M4_final_three_modal_fusion_validation_report.txt"
)

FLEXIBLE_REPORT_FILE = (
    EXISTING_FUSION_ROOT
    / "M4_final_flexible_multimodal_validation_report.txt"
)

required_files = [
    WEIGHTS_FILE,
    THREE_MODAL_RESULTS_FILE,
    FLEXIBLE_RESULTS_FILE,
    FINAL_CONFIG_FILE,
    THREE_MODAL_REPORT_FILE,
    FLEXIBLE_REPORT_FILE,
]

print("=" * 80)
print("CHECKING REQUIRED EXISTING ARTIFACTS")
print("=" * 80)

all_files_found = True

for path in required_files:
    exists = path.exists()

    print(
        f"{'FOUND' if exists else 'MISSING':>8} : {path.name}"
    )

    if not exists:
        all_files_found = False

if not all_files_found:
    raise FileNotFoundError(
        "One or more required existing fusion artifacts are missing."
    )

print("\nAll required artifacts found.")

CHECKING REQUIRED EXISTING ARTIFACTS
   FOUND : M3_1_fusion_weights.csv
   FOUND : M4_final_P001_P005_three_modal_fusion_results.csv
   FOUND : M4_final_P001_P005_flexible_multimodal_results.csv
   FOUND : M3_2_final_inference_config.json
   FOUND : M4_final_three_modal_fusion_validation_report.txt
   FOUND : M4_final_flexible_multimodal_validation_report.txt

All required artifacts found.


In [7]:
# ============================================================
# CELL 5 — LOAD VERIFIED FUSION WEIGHTS
# ============================================================

weights_df = pd.read_csv(WEIGHTS_FILE)

print("Fusion weight table:")
display(weights_df)

print("\nShape:", weights_df.shape)
print("Columns:", list(weights_df.columns))

Fusion weight table:


,Modality,Validation_ROC_AUC,Fusion_Weight
0,Genetic,0.788462,0.319370
1,Behavioral,0.988062,0.400219
2,Environmental,0.692281,0.280411



Shape: (3, 3)
Columns: ['Modality', 'Validation_ROC_AUC', 'Fusion_Weight']


In [8]:
# ============================================================
# CELL 6 — EXTRACT VERIFIED WEIGHTS
# ============================================================

required_modalities = [
    "Genetic",
    "Behavioral",
    "Environmental"
]

weight_map = {}

for modality in required_modalities:

    row = weights_df[
        weights_df["Modality"] == modality
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected exactly one weight for {modality}, "
            f"found {len(row)}."
        )

    weight_map[modality] = float(
        row["Fusion_Weight"].iloc[0]
    )

GENETIC_WEIGHT = weight_map["Genetic"]
BEHAVIORAL_WEIGHT = weight_map["Behavioral"]
ENVIRONMENTAL_WEIGHT = weight_map["Environmental"]

print("Verified fusion weights:")
print(f"Genetic       : {GENETIC_WEIGHT:.6f}")
print(f"Behavioral    : {BEHAVIORAL_WEIGHT:.6f}")
print(f"Environmental : {ENVIRONMENTAL_WEIGHT:.6f}")

Verified fusion weights:
Genetic       : 0.319370
Behavioral    : 0.400219
Environmental : 0.280411


In [9]:
# ============================================================
# CELL 7 — WEIGHT VALIDATION
# ============================================================

WEIGHT_SUM = (
    GENETIC_WEIGHT
    + BEHAVIORAL_WEIGHT
    + ENVIRONMENTAL_WEIGHT
)

print("Weight sum:", f"{WEIGHT_SUM:.9f}")

if not np.isclose(
    WEIGHT_SUM,
    1.0,
    atol=1e-6
):
    raise ValueError(
        f"Fusion weights do not sum to 1. "
        f"Observed sum = {WEIGHT_SUM}"
    )

print("\nWEIGHT VALIDATION: PASSED")

Weight sum: 1.000000000

WEIGHT VALIDATION: PASSED


In [10]:
# ============================================================
# CELL 8 — LOAD FINAL INFERENCE CONFIGURATION
# ============================================================

with open(
    FINAL_CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:
    final_config = json.load(f)

print(
    json.dumps(
        final_config,
        indent=2
    )
)

{
  "system": "ASD Multimodal Knowledge-Level Evidence Fusion",
  "version": "M3.2",
  "input_dimensions": {
    "genetic": 32,
    "behavioral": 32,
    "environmental": 32
  },
  "evidence_weights": {
    "genetic": 0.3194,
    "behavioral": 0.4002,
    "environmental": 0.2804
  },
  "decision_threshold": 0.5,
  "fusion_method": "weighted probability evidence fusion",
  "patient_matching_training": false,
  "row_wise_training_fusion": false,
  "single_patient_inference": true,
  "genetic_model": "/content/drive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_genetic_expert.pt",
  "behavioral_model": "/content/drive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_behavioral_expert.pt",
  "environmental_model": "/content/drive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt"
}


In [11]:
# ============================================================
# CELL 9 — CONFIGURATION VALIDATION
# ============================================================

expected_dimensions = {
    "genetic": 32,
    "behavioral": 32,
    "environmental": 32
}

actual_dimensions = final_config["input_dimensions"]

if actual_dimensions != expected_dimensions:
    raise ValueError(
        "Input dimensions do not match the verified configuration."
    )

if final_config["fusion_method"] != (
    "weighted probability evidence fusion"
):
    raise ValueError(
        "Unexpected fusion method."
    )

if final_config["patient_matching_training"] is not False:
    raise ValueError(
        "Patient matching training must be disabled."
    )

if final_config["row_wise_training_fusion"] is not False:
    raise ValueError(
        "Row-wise training fusion must be disabled."
    )

if final_config["single_patient_inference"] is not True:
    raise ValueError(
        "Single-patient inference must be enabled."
    )

DECISION_THRESHOLD = float(
    final_config["decision_threshold"]
)

if not np.isclose(
    DECISION_THRESHOLD,
    0.5
):
    raise ValueError(
        "Unexpected decision threshold."
    )

print("Configuration validation: PASSED")
print("Decision threshold:", DECISION_THRESHOLD)

Configuration validation: PASSED
Decision threshold: 0.5


In [12]:
# ============================================================
# CELL 10 — LOAD EXISTING THREE-MODALITY RESULTS
# ============================================================

three_modal_df = pd.read_csv(
    THREE_MODAL_RESULTS_FILE
)

print("Three-modality results:")
display(three_modal_df)

print("\nShape:", three_modal_df.shape)
print("\nColumns:")
for column in three_modal_df.columns:
    print(" -", column)

Three-modality results:


,patient_id,genetic_probability,behavioral_probability,environmental_probability,genetic_weight,behavioral_weight,environmental_weight,genetic_contribution,behavioral_contribution,environmental_contribution,final_probability,final_prediction,final_prediction_numeric
0,P001,0.519817,0.577138,0.080968,0.31937,0.400219,0.280411,0.166014,2.309816e-01,0.022704,0.419700,NON-ASD,0
1,P002,0.517192,0.000000,0.095078,0.31937,0.400219,0.280411,0.165176,0.000000e+00,0.026661,0.191837,NON-ASD,0
2,P003,0.515415,0.240194,0.048148,0.31937,0.400219,0.280411,0.164608,9.613020e-02,0.013501,0.274240,NON-ASD,0
3,P004,0.520541,0.000001,0.061390,0.31937,0.400219,0.280411,0.166245,4.002190e-07,0.017214,0.183460,NON-ASD,0
4,P005,0.517319,0.929673,0.085321,0.31937,0.400219,0.280411,0.165216,3.720728e-01,0.023925,0.561214,ASD,1



Shape: (5, 13)

Columns:
 - patient_id
 - genetic_probability
 - behavioral_probability
 - environmental_probability
 - genetic_weight
 - behavioral_weight
 - environmental_weight
 - genetic_contribution
 - behavioral_contribution
 - environmental_contribution
 - final_probability
 - final_prediction
 - final_prediction_numeric


In [13]:
# ============================================================
# CELL 11 — PROBABILITY INPUT VALIDATION
# ============================================================

probability_columns = [
    "genetic_probability",
    "behavioral_probability",
    "environmental_probability"
]

required_columns = [
    "patient_id",
    *probability_columns
]

missing_columns = [
    column
    for column in required_columns
    if column not in three_modal_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

for column in probability_columns:

    values = three_modal_df[column].to_numpy(
        dtype=float
    )

    if not np.isfinite(values).all():
        raise ValueError(
            f"Non-finite probability detected in {column}."
        )

    if (
        (values < 0).any()
        or
        (values > 1).any()
    ):
        raise ValueError(
            f"Probability outside [0,1] detected in {column}."
        )

print("Probability columns: PASSED")
print("NaN/Inf validation: PASSED")
print("Probability range [0,1]: PASSED")

Probability columns: PASSED
NaN/Inf validation: PASSED
Probability range [0,1]: PASSED


In [14]:
# ============================================================
# CELL 12 — CORE THREE-MODALITY FUSION FUNCTION
# ============================================================

def weighted_three_modal_fusion(
    genetic_probability,
    behavioral_probability,
    environmental_probability
):
    """
    Compute the finalized three-modality weighted
    probability evidence fusion.

    Formula:

        P_final =
            w_G * P_G
          + w_B * P_B
          + w_E * P_E
    """

    final_probability = (
        GENETIC_WEIGHT * genetic_probability
        +
        BEHAVIORAL_WEIGHT * behavioral_probability
        +
        ENVIRONMENTAL_WEIGHT * environmental_probability
    )

    return final_probability


print("Core three-modality fusion function defined.")

Core three-modality fusion function defined.


In [15]:
# ============================================================
# CELL 13 — COMPUTE MODALITY CONTRIBUTIONS
# ============================================================

fusion_df = three_modal_df[
    [
        "patient_id",
        "genetic_probability",
        "behavioral_probability",
        "environmental_probability"
    ]
].copy()

fusion_df["genetic_weight"] = GENETIC_WEIGHT
fusion_df["behavioral_weight"] = BEHAVIORAL_WEIGHT
fusion_df["environmental_weight"] = ENVIRONMENTAL_WEIGHT

fusion_df["genetic_contribution"] = (
    fusion_df["genetic_probability"]
    * fusion_df["genetic_weight"]
)

fusion_df["behavioral_contribution"] = (
    fusion_df["behavioral_probability"]
    * fusion_df["behavioral_weight"]
)

fusion_df["environmental_contribution"] = (
    fusion_df["environmental_probability"]
    * fusion_df["environmental_weight"]
)

fusion_df["final_probability"] = (
    fusion_df["genetic_contribution"]
    +
    fusion_df["behavioral_contribution"]
    +
    fusion_df["environmental_contribution"]
)

display(fusion_df)

,patient_id,genetic_probability,behavioral_probability,environmental_probability,genetic_weight,behavioral_weight,environmental_weight,genetic_contribution,behavioral_contribution,environmental_contribution,final_probability
0,P001,0.519817,0.577138,0.080968,0.31937,0.400219,0.280411,0.166014,2.309815e-01,0.022704,0.419700
1,P002,0.517192,0.000000,0.095078,0.31937,0.400219,0.280411,0.165175,0.000000e+00,0.026661,0.191836
2,P003,0.515415,0.240194,0.048148,0.31937,0.400219,0.280411,0.164608,9.613018e-02,0.013501,0.274239
3,P004,0.520541,0.000001,0.061390,0.31937,0.400219,0.280411,0.166245,4.002189e-07,0.017214,0.183460
4,P005,0.517319,0.929673,0.085321,0.31937,0.400219,0.280411,0.165216,3.720727e-01,0.023925,0.561214


In [16]:
# ============================================================
# CELL 14 — FINAL ASD / NON-ASD PREDICTION
# ============================================================

fusion_df["final_prediction_numeric"] = (
    fusion_df["final_probability"]
    >= DECISION_THRESHOLD
).astype(int)

fusion_df["final_prediction"] = np.where(
    fusion_df["final_prediction_numeric"] == 1,
    "ASD",
    "NON-ASD"
)

display(
    fusion_df[
        [
            "patient_id",
            "genetic_probability",
            "behavioral_probability",
            "environmental_probability",
            "final_probability",
            "final_prediction"
        ]
    ]
)

,patient_id,genetic_probability,behavioral_probability,environmental_probability,final_probability,final_prediction
0,P001,0.519817,0.577138,0.080968,0.419700,NON-ASD
1,P002,0.517192,0.000000,0.095078,0.191836,NON-ASD
2,P003,0.515415,0.240194,0.048148,0.274239,NON-ASD
3,P004,0.520541,0.000001,0.061390,0.183460,NON-ASD
4,P005,0.517319,0.929673,0.085321,0.561214,ASD


In [17]:
# ============================================================
# CELL 15 — THREE-MODALITY REGRESSION VALIDATION
# ============================================================

existing_three_modal = pd.read_csv(
    THREE_MODAL_RESULTS_FILE
)

comparison_df = fusion_df.merge(
    existing_three_modal[
        [
            "patient_id",
            "final_probability",
            "final_prediction_numeric"
        ]
    ],
    on="patient_id",
    suffixes=(
        "_new",
        "_existing"
    )
)

comparison_df["probability_difference"] = (
    comparison_df["final_probability_new"]
    -
    comparison_df["final_probability_existing"]
)

comparison_df["prediction_match"] = (
    comparison_df["final_prediction_numeric_new"]
    ==
    comparison_df["final_prediction_numeric_existing"]
)

display(comparison_df)

,patient_id,genetic_probability,behavioral_probability,environmental_probability,genetic_weight,behavioral_weight,environmental_weight,genetic_contribution,behavioral_contribution,environmental_contribution,final_probability_new,final_prediction_numeric_new,final_prediction,final_probability_existing,final_prediction_numeric_existing,probability_difference,prediction_match
0,P001,0.519817,0.577138,0.080968,0.31937,0.400219,0.280411,0.166014,2.309815e-01,0.022704,0.419700,0,NON-ASD,0.419700,0,-1.905238e-07,True
1,P002,0.517192,0.000000,0.095078,0.31937,0.400219,0.280411,0.165175,0.000000e+00,0.026661,0.191836,0,NON-ASD,0.191837,0,-1.186464e-07,True
2,P003,0.515415,0.240194,0.048148,0.31937,0.400219,0.280411,0.164608,9.613018e-02,0.013501,0.274239,0,NON-ASD,0.274240,0,-1.649005e-07,True
3,P004,0.520541,0.000001,0.061390,0.31937,0.400219,0.280411,0.166245,4.002189e-07,0.017214,0.183460,0,NON-ASD,0.183460,0,-1.338017e-07,True
4,P005,0.517319,0.929673,0.085321,0.31937,0.400219,0.280411,0.165216,3.720727e-01,0.023925,0.561214,1,ASD,0.561214,1,-2.277313e-07,True


In [18]:
# ============================================================
# CELL 16 — ASSERT THREE-MODAL REPRODUCTION
# ============================================================

max_probability_difference = (
    comparison_df["probability_difference"]
    .abs()
    .max()
)

all_predictions_match = (
    comparison_df["prediction_match"]
    .all()
)

print(
    "Maximum probability difference:",
    f"{max_probability_difference:.12f}"
)

print(
    "All predictions match:",
    all_predictions_match
)

if not np.isclose(
    max_probability_difference,
    0.0,
    atol=1e-6
):
    raise AssertionError(
        "Three-modality fusion does not reproduce "
        "the existing validated results."
    )

if not all_predictions_match:
    raise AssertionError(
        "Three-modality predictions do not match "
        "the existing validated results."
    )

print("\nTHREE-MODAL REPRODUCTION: PASSED")

Maximum probability difference: 0.000000227731
All predictions match: True

THREE-MODAL REPRODUCTION: PASSED


In [19]:
# ============================================================
# CELL 17 — FLEXIBLE MULTIMODAL FUSION
# ============================================================

ALL_WEIGHTS = {
    "Genetic": GENETIC_WEIGHT,
    "Behavioral": BEHAVIORAL_WEIGHT,
    "Environmental": ENVIRONMENTAL_WEIGHT
}


def flexible_weighted_fusion(
    probabilities
):
    """
    Perform weighted evidence fusion using only
    the modalities that are available.

    Parameters
    ----------
    probabilities : dict
        Dictionary such as:

        {
            "Genetic": 0.52,
            "Behavioral": 0.58
        }

    Returns
    -------
    float
        Renormalized weighted fusion probability.
    """

    if not probabilities:
        raise ValueError(
            "At least one modality probability is required."
        )

    available_modalities = list(
        probabilities.keys()
    )

    invalid_modalities = [
        modality
        for modality in available_modalities
        if modality not in ALL_WEIGHTS
    ]

    if invalid_modalities:
        raise ValueError(
            f"Unknown modalities: {invalid_modalities}"
        )

    probability_values = np.array(
        [
            float(probabilities[modality])
            for modality in available_modalities
        ],
        dtype=float
    )

    if not np.isfinite(probability_values).all():
        raise ValueError(
            "Probabilities must be finite."
        )

    if (
        (probability_values < 0).any()
        or
        (probability_values > 1).any()
    ):
        raise ValueError(
            "Probabilities must lie in [0,1]."
        )

    available_weights = np.array(
        [
            ALL_WEIGHTS[modality]
            for modality in available_modalities
        ],
        dtype=float
    )

    normalized_weights = (
        available_weights
        /
        available_weights.sum()
    )

    final_probability = float(
        np.sum(
            normalized_weights
            * probability_values
        )
    )

    return final_probability


print("Flexible fusion function defined.")

Flexible fusion function defined.


In [20]:
# ============================================================
# CELL 18 — FLEXIBLE FUSION EXAMPLES
# ============================================================

example_cases = {
    "Genetic": {
        "Genetic": 0.519817
    },

    "Behavioral": {
        "Behavioral": 0.577138
    },

    "Environmental": {
        "Environmental": 0.080968
    },

    "Genetic + Behavioral": {
        "Genetic": 0.519817,
        "Behavioral": 0.577138
    },

    "Genetic + Environmental": {
        "Genetic": 0.519817,
        "Environmental": 0.080968
    },

    "Behavioral + Environmental": {
        "Behavioral": 0.577138,
        "Environmental": 0.080968
    },

    "Genetic + Behavioral + Environmental": {
        "Genetic": 0.519817,
        "Behavioral": 0.577138,
        "Environmental": 0.080968
    }
}

for case_name, probabilities in example_cases.items():

    result = flexible_weighted_fusion(
        probabilities
    )

    prediction = (
        "ASD"
        if result >= DECISION_THRESHOLD
        else "NON-ASD"
    )

    print(
        f"{case_name:<35}"
        f" -> {result:.6f}"
        f" -> {prediction}"
    )

Genetic                             -> 0.519817 -> ASD
Behavioral                          -> 0.577138 -> ASD
Environmental                       -> 0.080968 -> NON-ASD
Genetic + Behavioral                -> 0.551698 -> ASD
Genetic + Environmental             -> 0.314645 -> NON-ASD
Behavioral + Environmental          -> 0.372722 -> NON-ASD
Genetic + Behavioral + Environmental -> 0.419700 -> NON-ASD


In [21]:
# ============================================================
# CELL 19 — LOAD EXISTING FLEXIBLE RESULTS
# ============================================================

existing_flexible_df = pd.read_csv(
    FLEXIBLE_RESULTS_FILE
)

print(
    "Existing flexible result shape:",
    existing_flexible_df.shape
)

print("\nColumns:")
for column in existing_flexible_df.columns:
    print(" -", column)

display(
    existing_flexible_df.head(10)
)

Existing flexible result shape: (35, 9)

Columns:
 - patient_id
 - available_modalities
 - num_modalities
 - genetic_probability
 - behavioral_probability
 - environmental_probability
 - final_probability
 - final_prediction
 - final_prediction_numeric


,patient_id,available_modalities,num_modalities,genetic_probability,behavioral_probability,environmental_probability,final_probability,final_prediction,final_prediction_numeric
0,P001,Genetic,1,0.519817,NaN,NaN,0.519817,ASD,1
1,P001,Behavioral,1,NaN,0.577138,NaN,0.577138,ASD,1
2,P001,Environmental,1,NaN,NaN,0.080968,0.080968,NON-ASD,0
3,P001,Genetic + Behavioral,2,0.519817,0.577138,NaN,0.551698,ASD,1
4,P001,Genetic + Environmental,2,0.519817,NaN,0.080968,0.314645,NON-ASD,0
5,P001,Behavioral + Environmental,2,NaN,0.577138,0.080968,0.372722,NON-ASD,0
6,P001,Genetic + Behavioral + Environmental,3,0.519817,0.577138,0.080968,0.419700,NON-ASD,0
7,P002,Genetic,1,0.517192,NaN,NaN,0.517192,ASD,1
8,P002,Behavioral,1,NaN,0.000000,NaN,0.000000,NON-ASD,0
9,P002,Environmental,1,NaN,NaN,0.095078,0.095078,NON-ASD,0


In [22]:
# ============================================================
# CELL 20 — REPRODUCE FLEXIBLE INFERENCE
# ============================================================

flexible_rows = []

for _, row in three_modal_df.iterrows():

    patient_id = row["patient_id"]

    probabilities = {
        "Genetic": float(
            row["genetic_probability"]
        ),
        "Behavioral": float(
            row["behavioral_probability"]
        ),
        "Environmental": float(
            row["environmental_probability"]
        )
    }

    modality_combinations = [
        ["Genetic"],
        ["Behavioral"],
        ["Environmental"],
        ["Genetic", "Behavioral"],
        ["Genetic", "Environmental"],
        ["Behavioral", "Environmental"],
        [
            "Genetic",
            "Behavioral",
            "Environmental"
        ]
    ]

    for available_modalities in modality_combinations:

        available_probabilities = {
            modality: probabilities[modality]
            for modality in available_modalities
        }

        final_probability = (
            flexible_weighted_fusion(
                available_probabilities
            )
        )

        final_prediction_numeric = int(
            final_probability
            >= DECISION_THRESHOLD
        )

        final_prediction = (
            "ASD"
            if final_prediction_numeric == 1
            else "NON-ASD"
        )

        flexible_rows.append(
            {
                "patient_id": patient_id,
                "available_modalities": (
                    " + ".join(
                        available_modalities
                    )
                ),
                "num_modalities": len(
                    available_modalities
                ),
                "genetic_probability": (
                    probabilities["Genetic"]
                    if "Genetic"
                    in available_modalities
                    else np.nan
                ),
                "behavioral_probability": (
                    probabilities["Behavioral"]
                    if "Behavioral"
                    in available_modalities
                    else np.nan
                ),
                "environmental_probability": (
                    probabilities["Environmental"]
                    if "Environmental"
                    in available_modalities
                    else np.nan
                ),
                "final_probability": final_probability,
                "final_prediction": final_prediction,
                "final_prediction_numeric": (
                    final_prediction_numeric
                )
            }
        )

flexible_df = pd.DataFrame(
    flexible_rows
)

print(
    "Generated flexible inference rows:",
    len(flexible_df)
)

display(
    flexible_df.head(10)
)

Generated flexible inference rows: 35


,patient_id,available_modalities,num_modalities,genetic_probability,behavioral_probability,environmental_probability,final_probability,final_prediction,final_prediction_numeric
0,P001,Genetic,1,0.519817,NaN,NaN,0.519817,ASD,1
1,P001,Behavioral,1,NaN,0.577138,NaN,0.577138,ASD,1
2,P001,Environmental,1,NaN,NaN,0.080968,0.080968,NON-ASD,0
3,P001,Genetic + Behavioral,2,0.519817,0.577138,NaN,0.551698,ASD,1
4,P001,Genetic + Environmental,2,0.519817,NaN,0.080968,0.314645,NON-ASD,0
5,P001,Behavioral + Environmental,2,NaN,0.577138,0.080968,0.372722,NON-ASD,0
6,P001,Genetic + Behavioral + Environmental,3,0.519817,0.577138,0.080968,0.419700,NON-ASD,0
7,P002,Genetic,1,0.517192,NaN,NaN,0.517192,ASD,1
8,P002,Behavioral,1,NaN,0.000000,NaN,0.000000,NON-ASD,0
9,P002,Environmental,1,NaN,NaN,0.095078,0.095078,NON-ASD,0


In [23]:
# ============================================================
# CELL 21 — FLEXIBLE STRUCTURE VALIDATION
# ============================================================

expected_row_count = 35

if len(flexible_df) != expected_row_count:
    raise AssertionError(
        f"Expected {expected_row_count} rows, "
        f"found {len(flexible_df)}."
    )

expected_combinations = {
    "Genetic",
    "Behavioral",
    "Environmental",
    "Genetic + Behavioral",
    "Genetic + Environmental",
    "Behavioral + Environmental",
    "Genetic + Behavioral + Environmental"
}

actual_combinations = set(
    flexible_df["available_modalities"]
)

if actual_combinations != expected_combinations:
    raise AssertionError(
        "Flexible modality combinations do not match "
        "the expected seven combinations."
    )

print("Row count validation: PASSED")
print("Combination validation: PASSED")

Row count validation: PASSED
Combination validation: PASSED


In [24]:
# ============================================================
# CELL 22 — FLEXIBLE FUSION REGRESSION VALIDATION
# ============================================================

comparison_columns = [
    "patient_id",
    "available_modalities",
    "final_probability",
    "final_prediction_numeric"
]

existing_compare = existing_flexible_df[
    comparison_columns
].copy()

new_compare = flexible_df[
    comparison_columns
].copy()

merged_flexible = new_compare.merge(
    existing_compare,
    on=[
        "patient_id",
        "available_modalities"
    ],
    suffixes=(
        "_new",
        "_existing"
    )
)

merged_flexible["probability_difference"] = (
    merged_flexible["final_probability_new"]
    -
    merged_flexible["final_probability_existing"]
)

merged_flexible["prediction_match"] = (
    merged_flexible[
        "final_prediction_numeric_new"
    ]
    ==
    merged_flexible[
        "final_prediction_numeric_existing"
    ]
)

display(
    merged_flexible
)

,patient_id,available_modalities,final_probability_new,final_prediction_numeric_new,final_probability_existing,final_prediction_numeric_existing,probability_difference,prediction_match
0,P001,Genetic,0.519817,1,0.519817,1,0.000000e+00,True
1,P001,Behavioral,0.577138,1,0.577138,1,0.000000e+00,True
2,P001,Environmental,0.080968,0,0.080968,0,0.000000e+00,True
3,P001,Genetic + Behavioral,0.551698,1,0.551698,1,9.588159e-09,True
4,P001,Genetic + Environmental,0.314645,0,0.314645,0,-2.682480e-07,True
5,P001,Behavioral + Environmental,0.372722,0,0.372722,0,-2.136820e-07,True
6,P001,Genetic + Behavioral + Environmental,0.419700,0,0.419700,0,-1.905238e-07,True
7,P002,Genetic,0.517192,1,0.517192,1,0.000000e+00,True
8,P002,Behavioral,0.000000,0,0.000000,0,0.000000e+00,True
9,P002,Environmental,0.095078,0,0.095078,0,0.000000e+00,True


In [25]:
# ============================================================
# CELL 23 — ASSERT FLEXIBLE REPRODUCTION
# ============================================================

max_flexible_difference = (
    merged_flexible[
        "probability_difference"
    ]
    .abs()
    .max()
)

all_flexible_predictions_match = (
    merged_flexible[
        "prediction_match"
    ]
    .all()
)

print(
    "Maximum flexible probability difference:",
    f"{max_flexible_difference:.12f}"
)

print(
    "All flexible predictions match:",
    all_flexible_predictions_match
)

if not np.isclose(
    max_flexible_difference,
    0.0,
    atol=1e-6
):
    raise AssertionError(
        "Flexible fusion does not reproduce "
        "the existing validated results."
    )

if not all_flexible_predictions_match:
    raise AssertionError(
        "Flexible predictions do not reproduce "
        "the existing validated results."
    )

print("\nFLEXIBLE FUSION REPRODUCTION: PASSED")

Maximum flexible probability difference: 0.000000363631
All flexible predictions match: True

FLEXIBLE FUSION REPRODUCTION: PASSED


In [26]:
# ============================================================
# CELL 24 — FINAL PROBABILITY RANGE VALIDATION
# ============================================================

all_three_probabilities = fusion_df[
    "final_probability"
].to_numpy(
    dtype=float
)

all_flexible_probabilities = flexible_df[
    "final_probability"
].to_numpy(
    dtype=float
)

combined_probabilities = np.concatenate(
    [
        all_three_probabilities,
        all_flexible_probabilities
    ]
)

if not np.isfinite(
    combined_probabilities
).all():
    raise AssertionError(
        "Non-finite fused probabilities detected."
    )

if (
    (combined_probabilities < 0).any()
    or
    (combined_probabilities > 1).any()
):
    raise AssertionError(
        "Fused probabilities outside [0,1]."
    )

print(
    "Fused probability minimum:",
    f"{combined_probabilities.min():.6f}"
)

print(
    "Fused probability maximum:",
    f"{combined_probabilities.max():.6f}"
)

print("\nFINAL PROBABILITY RANGE: PASSED")

Fused probability minimum: 0.000000
Fused probability maximum: 0.929673

FINAL PROBABILITY RANGE: PASSED


In [27]:
# ============================================================
# CELL 25 — FINAL THREE-MODALITY OUTPUT
# ============================================================

final_three_modal_df = fusion_df[
    [
        "patient_id",
        "genetic_probability",
        "behavioral_probability",
        "environmental_probability",
        "genetic_weight",
        "behavioral_weight",
        "environmental_weight",
        "genetic_contribution",
        "behavioral_contribution",
        "environmental_contribution",
        "final_probability",
        "final_prediction",
        "final_prediction_numeric"
    ]
].copy()

display(final_three_modal_df)

,patient_id,genetic_probability,behavioral_probability,environmental_probability,genetic_weight,behavioral_weight,environmental_weight,genetic_contribution,behavioral_contribution,environmental_contribution,final_probability,final_prediction,final_prediction_numeric
0,P001,0.519817,0.577138,0.080968,0.31937,0.400219,0.280411,0.166014,2.309815e-01,0.022704,0.419700,NON-ASD,0
1,P002,0.517192,0.000000,0.095078,0.31937,0.400219,0.280411,0.165175,0.000000e+00,0.026661,0.191836,NON-ASD,0
2,P003,0.515415,0.240194,0.048148,0.31937,0.400219,0.280411,0.164608,9.613018e-02,0.013501,0.274239,NON-ASD,0
3,P004,0.520541,0.000001,0.061390,0.31937,0.400219,0.280411,0.166245,4.002189e-07,0.017214,0.183460,NON-ASD,0
4,P005,0.517319,0.929673,0.085321,0.31937,0.400219,0.280411,0.165216,3.720727e-01,0.023925,0.561214,ASD,1


In [28]:
# ============================================================
# CELL 26 — FINAL FLEXIBLE OUTPUT
# ============================================================

final_flexible_df = flexible_df[
    [
        "patient_id",
        "available_modalities",
        "num_modalities",
        "genetic_probability",
        "behavioral_probability",
        "environmental_probability",
        "final_probability",
        "final_prediction",
        "final_prediction_numeric"
    ]
].copy()

display(
    final_flexible_df
)

,patient_id,available_modalities,num_modalities,genetic_probability,behavioral_probability,environmental_probability,final_probability,final_prediction,final_prediction_numeric
0,P001,Genetic,1,0.519817,NaN,NaN,0.519817,ASD,1
1,P001,Behavioral,1,NaN,0.577138,NaN,0.577138,ASD,1
2,P001,Environmental,1,NaN,NaN,0.080968,0.080968,NON-ASD,0
3,P001,Genetic + Behavioral,2,0.519817,0.577138,NaN,0.551698,ASD,1
4,P001,Genetic + Environmental,2,0.519817,NaN,0.080968,0.314645,NON-ASD,0
5,P001,Behavioral + Environmental,2,NaN,0.577138,0.080968,0.372722,NON-ASD,0
6,P001,Genetic + Behavioral + Environmental,3,0.519817,0.577138,0.080968,0.419700,NON-ASD,0
7,P002,Genetic,1,0.517192,NaN,NaN,0.517192,ASD,1
8,P002,Behavioral,1,NaN,0.000000,NaN,0.000000,NON-ASD,0
9,P002,Environmental,1,NaN,NaN,0.095078,0.095078,NON-ASD,0


In [29]:
# ============================================================
# CELL 27 — SAVE THREE-MODALITY RESULTS
# ============================================================

THREE_MODAL_OUTPUT = (
    REPORT_ROOT
    / "05_05_three_modal_fusion_results.csv"
)

final_three_modal_df.to_csv(
    THREE_MODAL_OUTPUT,
    index=False
)

print(
    "Saved:",
    THREE_MODAL_OUTPUT
)

print(
    "Rows:",
    len(final_three_modal_df)
)

Saved: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_05_multimodal_fusion\05_05_three_modal_fusion_results.csv
Rows: 5


In [30]:
# ============================================================
# CELL 28 — SAVE FLEXIBLE RESULTS
# ============================================================

FLEXIBLE_OUTPUT = (
    REPORT_ROOT
    / "05_05_flexible_multimodal_results.csv"
)

final_flexible_df.to_csv(
    FLEXIBLE_OUTPUT,
    index=False
)

print(
    "Saved:",
    FLEXIBLE_OUTPUT
)

print(
    "Rows:",
    len(final_flexible_df)
)

Saved: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_05_multimodal_fusion\05_05_flexible_multimodal_results.csv
Rows: 35


In [31]:
# ============================================================
# CELL 29 — BUILD FINAL JSON REPORT
# ============================================================

report = {
    "phase": "05.05",
    "name": "Multimodal Fusion",
    "status": "PASSED",

    "fusion_method": (
        "knowledge-level weighted probability evidence fusion"
    ),

    "training_performed": False,
    "checkpoint_modified": False,
    "row_wise_training_fusion": False,
    "patient_matching_training": False,
    "single_patient_inference": True,

    "input_dimensions": {
        "genetic": 32,
        "behavioral": 32,
        "environmental": 32
    },

    "fusion_weights": {
        "genetic": GENETIC_WEIGHT,
        "behavioral": BEHAVIORAL_WEIGHT,
        "environmental": ENVIRONMENTAL_WEIGHT
    },

    "weight_sum": WEIGHT_SUM,

    "decision_threshold": DECISION_THRESHOLD,

    "three_modal_validation": {
        "rows": len(final_three_modal_df),
        "max_probability_difference": (
            float(max_probability_difference)
        ),
        "predictions_match": bool(
            all_predictions_match
        ),
        "status": "PASSED"
    },

    "flexible_validation": {
        "rows": len(final_flexible_df),
        "max_probability_difference": (
            float(max_flexible_difference)
        ),
        "predictions_match": bool(
            all_flexible_predictions_match
        ),
        "status": "PASSED"
    },

    "flexible_modes": [
        "ANY-ONE",
        "ANY-TWO",
        "ALL-THREE"
    ],

    "outputs": {
        "three_modal_csv": str(
            THREE_MODAL_OUTPUT
        ),
        "flexible_csv": str(
            FLEXIBLE_OUTPUT
        )
    }
}

REPORT_OUTPUT = (
    REPORT_ROOT
    / "05_05_multimodal_fusion_report.json"
)

with open(
    REPORT_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2
    )

print(
    "Saved report:",
    REPORT_OUTPUT
)

Saved report: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_05_multimodal_fusion\05_05_multimodal_fusion_report.json


In [32]:
# ============================================================
# CELL 30 — FINAL ARTIFACT VERIFICATION
# ============================================================

expected_outputs = [
    THREE_MODAL_OUTPUT,
    FLEXIBLE_OUTPUT,
    REPORT_OUTPUT
]

print("=" * 80)
print("05.05 FINAL ARTIFACT VERIFICATION")
print("=" * 80)

all_outputs_exist = True

for output_path in expected_outputs:

    exists = output_path.exists()

    print(
        f"{'FOUND' if exists else 'MISSING':>8} : "
        f"{output_path.name}"
    )

    if not exists:
        all_outputs_exist = False

if not all_outputs_exist:
    raise FileNotFoundError(
        "One or more 05.05 output artifacts are missing."
    )

# Re-read CSV outputs to verify they are valid
verified_three_modal = pd.read_csv(
    THREE_MODAL_OUTPUT
)

verified_flexible = pd.read_csv(
    FLEXIBLE_OUTPUT
)

if len(verified_three_modal) != 5:
    raise AssertionError(
        "Three-modality output must contain 5 rows."
    )

if len(verified_flexible) != 35:
    raise AssertionError(
        "Flexible output must contain 35 rows."
    )

with open(
    REPORT_OUTPUT,
    "r",
    encoding="utf-8"
) as f:
    verified_report = json.load(f)

if verified_report["status"] != "PASSED":
    raise AssertionError(
        "Final JSON report status is not PASSED."
    )

print("\nCSV reload verification: PASSED")
print("JSON reload verification: PASSED")
print("Output count verification: PASSED")

05.05 FINAL ARTIFACT VERIFICATION
   FOUND : 05_05_three_modal_fusion_results.csv
   FOUND : 05_05_flexible_multimodal_results.csv
   FOUND : 05_05_multimodal_fusion_report.json

CSV reload verification: PASSED
JSON reload verification: PASSED
Output count verification: PASSED


In [33]:
# ============================================================
# CELL 31 — FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("PHASE 05.05 — MULTIMODAL FUSION")
print("=" * 80)

print("\nSTATUS: PASSED")

print("\nFusion method:")
print("Knowledge-level weighted probability evidence fusion")

print("\nVerified weights:")
print(f"Genetic       : {GENETIC_WEIGHT:.6f}")
print(f"Behavioral    : {BEHAVIORAL_WEIGHT:.6f}")
print(f"Environmental : {ENVIRONMENTAL_WEIGHT:.6f}")

print(
    f"Weight sum    : {WEIGHT_SUM:.6f}"
)

print(
    f"Decision threshold: {DECISION_THRESHOLD:.2f}"
)

print("\nValidation:")
print("Three-modality reproduction : PASSED")
print("Flexible fusion reproduction: PASSED")
print("Probability range           : PASSED")
print("Artifact verification       : PASSED")

print("\nTraining performed:")
print("NO")

print("\nCheckpoint modified:")
print("NO")

print("\nRow-wise patient fusion training:")
print("NO")

print("\nPatient matching performed:")
print("NO")

print("\nOutputs:")
print(" - 05_05_three_modal_fusion_results.csv")
print(" - 05_05_flexible_multimodal_results.csv")
print(" - 05_05_multimodal_fusion_report.json")

print("\nNext phase:")
print("05.06 — Integration Validation / Final Phase 05 Verification")

print("\n" + "=" * 80)
print("PHASE 05.05 COMPLETE")
print("=" * 80)


PHASE 05.05 — MULTIMODAL FUSION

STATUS: PASSED

Fusion method:
Knowledge-level weighted probability evidence fusion

Verified weights:
Genetic       : 0.319370
Behavioral    : 0.400219
Environmental : 0.280411
Weight sum    : 1.000000
Decision threshold: 0.50

Validation:
Three-modality reproduction : PASSED
Flexible fusion reproduction: PASSED
Probability range           : PASSED
Artifact verification       : PASSED

Training performed:
NO

Checkpoint modified:
NO

Row-wise patient fusion training:
NO

Patient matching performed:
NO

Outputs:
 - 05_05_three_modal_fusion_results.csv
 - 05_05_flexible_multimodal_results.csv
 - 05_05_multimodal_fusion_report.json

Next phase:
05.06 — Integration Validation / Final Phase 05 Verification

PHASE 05.05 COMPLETE
